# Week 7 Assignment Demo  
## Temperature Trends Using NOAA Climate Data

This notebook demonstrates a basic time-series workflow using NOAA Climate-at-a-Glance temperature departure data.

It now supports the NOAA format that looks like this:

```text
# Title: Global Land and Ocean 12-Month Period Average Temperature Departures
# Units: Degrees Celsius
# Base Period: 1901-2000
Date    Departure from Average
189501  -0.32
189502  -0.32
...
```

The column **Departure from Average** is the temperature anomaly.  
The date format `YYYYMM` is converted into a year.

## Learning goals

By the end of this assignment, you should be able to:

1. Load and inspect temperature time-series data.
2. Explain what a temperature anomaly or departure from average means.
3. Identify time-series concepts such as trend, variability, anomaly, and forecasting.
4. Visualize temperature departures over time.
5. Aggregate monthly or 12-month period data to annual values.
6. Use a moving average to reduce short-term variability.
7. Fit a simple trend line.
8. Interpret the usefulness and limitations of simple AI/data-driven climate analysis for environmental decision-making.

## Important note about NOAA “Departure from Average”

In NOAA Climate-at-a-Glance, **Departure from Average** means the same thing as a temperature anomaly.

For this assignment:

```text
Departure from Average = temperature anomaly
```

If the NOAA file says:

```text
Units: Degrees Celsius
```

then the departure values are already in degrees Celsius.

If the NOAA file says:

```text
Units: Degrees Fahrenheit
```

then the values should be converted to Celsius.

## Step 1: Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## Step 2: Load the NOAA data

Use one of the following options:

1. Paste a raw GitHub link to a NOAA or course CSV/TSV file.
2. Upload a NOAA file to Colab.
3. Upload the course sample file.

The loader can handle:

- simplified course CSV with `year` and `temperature_anomaly_c`,
- NOAA tab-delimited files with `Date` and `Departure from Average`,
- comment lines beginning with `#`,
- monthly or 12-month-period date values such as `189501`.

In [ ]:
# Option 1: Paste a raw GitHub link here, if available.
data_url = ""  # example: "https://raw.githubusercontent.com/.../noaa_temperature_departures.tsv"

# Option 2: Use an uploaded file name.
uploaded_filename = "noaa_global_land_ocean_12month_departures_sample.tsv"

def detect_units(path_or_url):
    """Read NOAA comment lines and detect Celsius/Fahrenheit if stated."""
    try:
        if str(path_or_url).startswith("http"):
            import requests
            text = requests.get(path_or_url).text.splitlines()
        else:
            with open(path_or_url, "r", encoding="utf-8") as f:
                text = f.read().splitlines()
    except Exception:
        return "unknown"

    for line in text[:20]:
        lower = line.lower()
        if "units" in lower and "celsius" in lower:
            return "c"
        if "units" in lower and "fahrenheit" in lower:
            return "f"
    return "unknown"

def load_noaa_temperature_data(path_or_url, aggregate_to_annual=True):
    """Load NOAA Climate-at-a-Glance style data or simplified course CSV."""
    units = detect_units(path_or_url)

    # Try automatic separator detection. comment="#" skips NOAA metadata lines.
    raw = pd.read_csv(path_or_url, comment="#", sep=None, engine="python")
    raw.columns = [str(c).strip() for c in raw.columns]
    lower_cols = {c.lower(): c for c in raw.columns}

    # Case 1: simplified course format
    if "year" in lower_cols and "temperature_anomaly_c" in lower_cols:
        out = raw[[lower_cols["year"], lower_cols["temperature_anomaly_c"]]].copy()
        out.columns = ["year", "temperature_anomaly_c"]
        out["year"] = pd.to_numeric(out["year"], errors="coerce").astype("Int64")
        out["temperature_anomaly_c"] = pd.to_numeric(out["temperature_anomaly_c"], errors="coerce")
        return out.dropna().sort_values("year").reset_index(drop=True)

    # Case 2: NOAA format: Date + Departure from Average
    date_col = None
    value_col = None

    for c in raw.columns:
        clean = c.strip().lower()
        if clean == "date" or clean == "year":
            date_col = c
        if "departure" in clean or "anomaly" in clean or clean == "value":
            value_col = c

    if date_col is None:
        date_col = raw.columns[0]
    if value_col is None:
        # Pick the first numeric-looking column after the date column.
        for c in raw.columns:
            if c != date_col and pd.to_numeric(raw[c], errors="coerce").notna().sum() > 0:
                value_col = c
                break

    out = pd.DataFrame()
    date_num = pd.to_numeric(raw[date_col], errors="coerce")
    out["date"] = date_num.astype("Int64")

    # NOAA YYYYMM format: 189501 = Jan 1895.
    out["year"] = np.where(date_num > 10000, np.floor(date_num / 100), date_num)
    out["year"] = pd.to_numeric(out["year"], errors="coerce")

    out["departure_from_average"] = pd.to_numeric(raw[value_col], errors="coerce")

    # Convert to Celsius only if the file says Fahrenheit.
    if units == "f":
        out["temperature_anomaly_c"] = out["departure_from_average"] * 5/9
    else:
        out["temperature_anomaly_c"] = out["departure_from_average"]

    out = out.dropna(subset=["year", "temperature_anomaly_c"]).copy()
    out["year"] = out["year"].astype(int)

    if aggregate_to_annual:
        annual = (
            out.groupby("year", as_index=False)
            .agg(temperature_anomaly_c=("temperature_anomaly_c", "mean"))
        )
        annual["temperature_anomaly_c"] = annual["temperature_anomaly_c"].round(3)
        return annual.sort_values("year").reset_index(drop=True)

    return out.sort_values(["year", "date"]).reset_index(drop=True)

if data_url:
    temp_df = load_noaa_temperature_data(data_url, aggregate_to_annual=True)
else:
    temp_df = load_noaa_temperature_data(uploaded_filename, aggregate_to_annual=True)

temp_df.head()

## Step 3: Inspect the data

After loading, the notebook should produce a simplified annual table with:

- `year`
- `temperature_anomaly_c`

If the original NOAA file had monthly or 12-month-period rows, the notebook aggregates them to annual averages.

In [ ]:
print("Rows and columns:", temp_df.shape)
print("Year range:", int(temp_df["year"].min()), "to", int(temp_df["year"].max()))
print("\nFirst rows:")
display(temp_df.head())

print("\nLast rows:")
display(temp_df.tail())

print("\nSummary:")
print(temp_df["temperature_anomaly_c"].describe())

## Step 4: Plot the temperature anomaly time series

This plot shows annual temperature departures from average over time.

Look for:

- long-term trend,
- year-to-year variability,
- unusually warm or cool periods,
- recent changes compared with earlier years.

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(temp_df["year"], temp_df["temperature_anomaly_c"], marker="o", linewidth=1, markersize=3)
plt.axhline(0, linestyle="--", linewidth=1)
plt.xlabel("Year")
plt.ylabel("Temperature departure from average (°C)")
plt.title("Annual Temperature Departure from Average")
plt.show()

## Step 5: Add a moving average

A moving average smooths short-term variability so the longer-term pattern is easier to see.

Here we use a 10-year moving average.

In [ ]:
temp_df["moving_average_10yr"] = temp_df["temperature_anomaly_c"].rolling(window=10, center=True).mean()

plt.figure(figsize=(10, 5))
plt.plot(temp_df["year"], temp_df["temperature_anomaly_c"], marker="o", linewidth=0.8, markersize=3, label="Annual departure")
plt.plot(temp_df["year"], temp_df["moving_average_10yr"], linewidth=2.5, label="10-year moving average")
plt.axhline(0, linestyle="--", linewidth=1)
plt.xlabel("Year")
plt.ylabel("Temperature departure from average (°C)")
plt.title("Annual Temperature Departures with 10-Year Moving Average")
plt.legend()
plt.show()

## Step 6: Fit a simple trend line

This is a simple data-driven model using year to estimate temperature departure from average.

This is not a complete climate model. It is a basic way to summarize the direction and rate of change in a time series.

In [ ]:
X = temp_df[["year"]]
y = temp_df["temperature_anomaly_c"]

trend_model = LinearRegression()
trend_model.fit(X, y)

temp_df["trend_line"] = trend_model.predict(X)

slope_per_year = trend_model.coef_[0]
slope_per_decade = slope_per_year * 10

print("Estimated trend:", round(slope_per_decade, 3), "°C per decade")
print("Intercept:", round(trend_model.intercept_, 3))

In [ ]:
plt.figure(figsize=(10, 5))
plt.scatter(temp_df["year"], temp_df["temperature_anomaly_c"], s=18, label="Annual departure")
plt.plot(temp_df["year"], temp_df["trend_line"], linewidth=2.5, label="Linear trend")
plt.axhline(0, linestyle="--", linewidth=1)
plt.xlabel("Year")
plt.ylabel("Temperature departure from average (°C)")
plt.title("Temperature Departure Trend")
plt.legend()
plt.show()

## Step 7: Compare early and recent periods

Here we compare:

- the first 30 years in the dataset,
- the most recent 30 years in the dataset.

In [ ]:
n_years = min(30, len(temp_df) // 3)

first_period = temp_df.head(n_years)
recent_period = temp_df.tail(n_years)

first_mean = first_period["temperature_anomaly_c"].mean()
recent_mean = recent_period["temperature_anomaly_c"].mean()
difference = recent_mean - first_mean

comparison = pd.DataFrame([
    {"Period": f"{int(first_period['year'].min())}-{int(first_period['year'].max())}", "Mean departure (°C)": first_mean},
    {"Period": f"{int(recent_period['year'].min())}-{int(recent_period['year'].max())}", "Mean departure (°C)": recent_mean},
    {"Period": "Difference", "Mean departure (°C)": difference}
])

comparison.round(3)

## Step 8: Simple forecasting example

A linear model can be extended into the near future, but this should be interpreted carefully.

This is only a simple demonstration of forecasting logic. It does not include physical climate processes, uncertainty ranges, emissions scenarios, or climate model ensembles.

In [ ]:
future_years = pd.DataFrame({"year": np.arange(int(temp_df["year"].max()) + 1, int(temp_df["year"].max()) + 11)})
future_years["forecast_anomaly_c"] = trend_model.predict(future_years[["year"]])

future_years.round(3)

In [ ]:
plt.figure(figsize=(10, 5))
plt.scatter(temp_df["year"], temp_df["temperature_anomaly_c"], s=18, label="Observed annual departure")
plt.plot(temp_df["year"], temp_df["trend_line"], linewidth=2, label="Trend fit")
plt.plot(future_years["year"], future_years["forecast_anomaly_c"], marker="o", linestyle="--", label="Simple forecast")
plt.axhline(0, linestyle="--", linewidth=1)
plt.xlabel("Year")
plt.ylabel("Temperature departure from average (°C)")
plt.title("Simple Temperature Trend Forecast")
plt.legend()
plt.show()

## Step 9: Optional model evaluation

This simple test uses earlier years for training and later years for testing.

This is more appropriate for a time series than randomly shuffling all years.

In [ ]:
split_year = int(temp_df["year"].quantile(0.75))

train_df = temp_df[temp_df["year"] <= split_year].copy()
test_df = temp_df[temp_df["year"] > split_year].copy()

eval_model = LinearRegression()
eval_model.fit(train_df[["year"]], train_df["temperature_anomaly_c"])

test_df["predicted_anomaly_c"] = eval_model.predict(test_df[["year"]])

mae = mean_absolute_error(test_df["temperature_anomaly_c"], test_df["predicted_anomaly_c"])
rmse = mean_squared_error(test_df["temperature_anomaly_c"], test_df["predicted_anomaly_c"]) ** 0.5
r2 = r2_score(test_df["temperature_anomaly_c"], test_df["predicted_anomaly_c"])

print("Train through year:", split_year)
print("Test years:", int(test_df["year"].min()), "to", int(test_df["year"].max()))
print("MAE:", round(mae, 3), "°C")
print("RMSE:", round(rmse, 3), "°C")
print("R²:", round(r2, 3))

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(train_df["year"], train_df["temperature_anomaly_c"], marker="o", markersize=3, linewidth=1, label="Training data")
plt.plot(test_df["year"], test_df["temperature_anomaly_c"], marker="o", markersize=3, linewidth=1, label="Test observed")
plt.plot(test_df["year"], test_df["predicted_anomaly_c"], marker="o", linestyle="--", label="Test predicted")
plt.xlabel("Year")
plt.ylabel("Temperature departure from average (°C)")
plt.title("Simple Time-Series Model Evaluation")
plt.legend()
plt.show()

# Orange workflow notes

The same NOAA file can be used in Orange, but it is easier if the instructor provides a cleaned CSV with:

- `year`
- `temperature_anomaly_c`

For the NOAA raw file, the `Date` column is in `YYYYMM` format. In Orange, students may find it easier to use the instructor-cleaned annual CSV.

## Basic Orange visualization workflow

`File → Data Table → Line Plot`

Use:

- x-axis: `year`
- y-axis: `temperature_anomaly_c`

You may also use:

`File → Scatter Plot`

Use:

- x-axis: `year`
- y-axis: `temperature_anomaly_c`

## Modeling workflow

`File → Select Columns → Linear Regression → Predictions`

In **Select Columns**:

- Feature: `year`
- Target: `temperature_anomaly_c`

# Reflection questions

Answer these in your assignment report.

1. What dataset did you use, and what years are included?
2. What does “Departure from Average” mean?
3. What long-term trend do you see?
4. What year-to-year variability do you see?
5. What does the 10-year moving average show?
6. What is the estimated trend in °C per decade?
7. How does the recent period compare with the earliest period?
8. What does the simple forecast suggest?
9. Why should this forecast be interpreted carefully?
10. What are the limitations of using a simple linear model for climate decision-making?